In [ ]:
"""
KoChatGPT 데이터셋 EDA 워크북
========================================

【 대상 데이터 】
- kochatgpt_1_RM.jsonl (보상 모델용)

RM 데이터의 구조:
- prompt: 질문
- 여러 개의 completion (보통 3~4개)
- ranking 또는 score (어느 것이 좋은지 표시)

분석 목표:
1. 데이터 구조 확인
2. ranking/score 일관성 확인
3. completion 품질 분포
4. 오류 탐지
"""

In [1]:
# === Colab 호환 셋업 (자동 추가) ===
import torch as _t
_t._orig_load = getattr(_t, '_orig_load', _t.load)
def _compat_load(*a, **k):
    k.setdefault('weights_only', False)
    return _t._orig_load(*a, **k)
_t.load = _compat_load
try:
    import matplotlib; matplotlib.rcParams['axes.unicode_minus'] = False
except Exception: pass
print('[colab-compat] torch.load weights_only=False 패치')


[colab-compat] torch.load weights_only=False 패치


In [2]:
!git clone https://github.com/airobotlab/KoChatGPT
!cp -r /content/KoChatGPT/colossalai_ChatGPT_230319/chatgpt /content/chatgpt

Cloning into 'KoChatGPT'...
remote: Enumerating objects: 304, done.
remote: Total 304 (delta 0), reused 0 (delta 0), pack-reused 304 (from 1)
Receiving objects: 100% (304/304), 57.72 MiB | 24.95 MiB/s, done.
Resolving deltas: 100% (123/123), done.


In [1]:
import json
from collections import Counter
import statistics

# ============================================================================
# 데이터 로드 및 구조 확인
# ============================================================================

print("="*80)
print("RM (Reward Model) 데이터 EDA")
print("="*80)

print("\n【 1단계: 데이터 로드 및 기본 통계 】\n")

try:
    with open('/content/KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl', 'r', encoding='utf-8') as f:
        rm_data = json.load(f)
    print(f"✓ kochatgpt_1_RM.jsonl 로드 성공")
    print(f"  총 {len(rm_data)}건의 레코드\n")
except FileNotFoundError:
    print("✗ 파일을 찾을 수 없음: kochatgpt_1_RM.jsonl")
    rm_data = []
except json.JSONDecodeError as e:
    print(f"✗ JSON 파싱 오류: {e}")
    rm_data = []

if not rm_data:
    print("데이터가 없어서 분석 불가능")
    exit()

# ============================================================================
# 구조 확인
# ============================================================================

print("【 2단계: 데이터 구조 분석 】\n")

# 첫 몇 개 샘플 확인
print("첫 1개 레코드 구조:\n")
first_record = rm_data[0]
print(f"Keys: {list(first_record.keys())}\n")

for key in first_record.keys():
    value = first_record[key]
    if isinstance(value, str):
        print(f"  {key}: {repr(value[:80])}...")
    elif isinstance(value, (list, dict)):
        print(f"  {key}: {type(value).__name__} (길이: {len(value)})")
    else:
        print(f"  {key}: {value}")

# ============================================================================
# 구조별 통계
# ============================================================================

print("\n" + "="*80)
print("【 3단계: Completion 개수 분포 】\n")

# 각 레코드가 가진 completion 개수 파악
completion_count_patterns = {}

for record in rm_data:
    # completion 필드 탐색 (completion, completion_a, completion_1 등 가능성)
    completion_keys = [k for k in record.keys() if 'completion' in k.lower()]
    num_completions = len(completion_keys)

    if num_completions not in completion_count_patterns:
        completion_count_patterns[num_completions] = []
    completion_count_patterns[num_completions].append(record)

for num, records in sorted(completion_count_patterns.items()):
    print(f"  {num}개 completion 레코드: {len(records)}건 ({len(records)/len(rm_data)*100:.1f}%)")

# ============================================================================
# Ranking/Score 구조 확인
# ============================================================================

print("\n" + "="*80)
print("【 4단계: Ranking/Score 구조 분석 】\n")

ranking_patterns = {}
score_patterns = {}

for record in rm_data:
    # ranking, scores, rank, rating 등 다양한 필드명 가능
    ranking_keys = [k for k in record.keys() if 'rank' in k.lower() or 'score' in k.lower()]

    for key in ranking_keys:
        value = record[key]
        if isinstance(value, (list, tuple)):
            ranking_patterns[key] = ranking_patterns.get(key, 0) + 1
        elif isinstance(value, dict):
            score_patterns[key] = score_patterns.get(key, 0) + 1

print("Ranking/Score 필드 분포:")
for key, count in sorted(ranking_patterns.items()):
    print(f"  {key} (list): {count}건")
for key, count in sorted(score_patterns.items()):
    print(f"  {key} (dict): {count}건")

# ============================================================================
# Score 분석 (순서 기반이 아닌 실제 점수 기반)
# ============================================================================

print("\n" + "="*80)
print("【 5단계: Score 분석 】\n")

# 첫 번째 ranking/score 필드 선택
ranking_field = None
for key in rm_data[0].keys():
    if 'rank' in key.lower() or 'score' in key.lower():
        ranking_field = key
        break

if ranking_field:
    print(f"분석 대상 필드: '{ranking_field}'\n")

    # Score 값 수집
    scores_sample = []
    score_sequences = []

    for record in rm_data[:100]:  # 처음 100개 샘플
        ranking = record.get(ranking_field)
        if isinstance(ranking, (list, tuple)):
            scores_sample.extend(ranking)
            score_sequences.append(ranking)

    if scores_sample:
        print(f"Score 범위: {min(scores_sample)} ~ {max(scores_sample)}")
        print(f"평균: {statistics.mean(scores_sample):.1f}")
        print(f"중앙값: {statistics.median(scores_sample):.1f}")
        print(f"표준편차: {statistics.stdev(scores_sample) if len(scores_sample) > 1 else 0:.1f}\n")

    # ✅ 의미있는 분석: Score 분포
    print("【 Score 분포 분석 】\n")

    # 1. Score 간격 분석 (각 completion 간 점수 차이)
    print("각 레코드 내 score 차이 분석:")
    score_diffs = []
    for seq in score_sequences:
        if len(seq) >= 2:
            max_diff = max(seq) - min(seq)
            score_diffs.append(max_diff)

    if score_diffs:
        print(f"  • 최대 차이 평균: {statistics.mean(score_diffs):.2f}")
        print(f"  • 최대 차이 범위: {min(score_diffs)} ~ {max(score_diffs)}\n")

        # 차이가 0인 경우 (구분 불가)
        zero_diff = sum(1 for d in score_diffs if d == 0)
        print(f"  • 차이가 0인 경우 (구분 불가): {zero_diff}건 ({zero_diff/len(score_diffs)*100:.1f}%)")
        print(f"    → 이런 경우는 정제 필요할 수 있음\n")

    # 2. Score 유형 분석
    print("【 Score 유형 분석 】\n")

    # 정수 vs 실수
    int_scores = sum(1 for s in scores_sample if isinstance(s, int))
    float_scores = sum(1 for s in scores_sample if isinstance(s, float))

    print(f"  • 정수형: {int_scores}건 ({int_scores/len(scores_sample)*100:.1f}%)")
    print(f"  • 실수형: {float_scores}건 ({float_scores/len(scores_sample)*100:.1f}%)")

    # Score 범주 (1-5, 0-10, 0-100 등)
    score_range = max(scores_sample) - min(scores_sample)
    if score_range <= 5:
        print(f"  • 범위 유형: 1-5점 척도 (범위: {score_range})")
    elif score_range <= 10:
        print(f"  • 범위 유형: 0-10점 척도 (범위: {score_range})")
    elif score_range <= 100:
        print(f"  • 범위 유형: 0-100점 척도 (범위: {score_range})")
    else:
        print(f"  • 범위 유형: 기타 (범위: {score_range})")

    print()

    # 3. 중요한 검사: Score 일관성
    print("【 Score 일관성 검사 】\n")

    # Score가 모두 같은 경우
    all_same = sum(1 for seq in score_sequences if len(set(seq)) == 1)
    print(f"  • 모든 completion의 score가 같은 경우: {all_same}건")
    print(f"    → 학습에 도움이 되지 않으므로 제거 권장\n")

    # Score가 중복되는 경우 (예: [5, 5, 3])
    has_duplicate = sum(1 for seq in score_sequences if len(seq) != len(set(seq)))
    print(f"  • Score에 중복이 있는 경우: {has_duplicate}건")
    print(f"    → 보상 모델 학습이 약할 수 있음\n")

else:
    print("Ranking/Score 필드를 찾을 수 없음\n")

# ============================================================================
# Completion 품질 분포
# ============================================================================

print("="*80)
print("【 6단계: Completion 길이 분포 】\n")

completion_lengths = []
for record in rm_data:
    completion_keys = [k for k in record.keys() if 'completion' in k.lower()]
    for key in completion_keys:
        comp = record[key]
        if isinstance(comp, str):
            completion_lengths.append(len(comp))

if completion_lengths:
    print(f"총 {len(completion_lengths)}개 completion 분석\n")
    print(f"길이 분포:")
    print(f"  최소: {min(completion_lengths)}자")
    print(f"  최대: {max(completion_lengths)}자")
    print(f"  평균: {statistics.mean(completion_lengths):.1f}자")
    print(f"  중앙값: {statistics.median(completion_lengths):.1f}자")
    print(f"  표준편차: {statistics.stdev(completion_lengths):.1f}자\n")

# ============================================================================
# 오류 탐지
# ============================================================================

print("="*80)
print("【 7단계: 오류 탐지 】\n")

errors = {
    'missing_prompt': 0,
    'missing_completion': 0,
    'missing_ranking': 0,
    'empty_completion': 0,
    'ranking_mismatch': 0,
    'parsing_error': 0,
}

for idx, record in enumerate(rm_data):
    # prompt 확인
    if 'prompt' not in record or not record['prompt']:
        errors['missing_prompt'] += 1

    # completion 확인
    completion_keys = [k for k in record.keys() if 'completion' in k.lower()]
    if not completion_keys:
        errors['missing_completion'] += 1
    else:
        for key in completion_keys:
            if not record[key] or not isinstance(record[key], str):
                errors['empty_completion'] += 1

    # ranking 확인
    ranking_keys = [k for k in record.keys() if 'rank' in k.lower() or 'score' in k.lower()]
    if not ranking_keys:
        errors['missing_ranking'] += 1
    else:
        for key in ranking_keys:
            value = record[key]
            if isinstance(value, (list, tuple)):
                if len(value) != len(completion_keys):
                    errors['ranking_mismatch'] += 1

    # 파싱 오류 (dict 구조 섞임)
    if any('token' in str(v) for v in record.values() if isinstance(v, str)):
        errors['parsing_error'] += 1

print("발견된 오류:")
for error_type, count in errors.items():
    if count > 0:
        print(f"  {error_type}: {count}건 ({count/len(rm_data)*100:.1f}%)")

if sum(errors.values()) == 0:
    print("  오류 없음 ✓\n")
else:
    print()

# ============================================================================
# 샘플 보기
# ============================================================================

print("="*80)
print("【 8단계: 데이터 샘플 】\n")

print("처음 3개 레코드:\n")
for i, record in enumerate(rm_data[:3], 1):
    print(f"레코드 {i}:")
    print(f"  prompt: {record.get('prompt', 'N/A')[:80]}")

    completion_keys = [k for k in record.keys() if 'completion' in k.lower()]
    for j, key in enumerate(completion_keys, 1):
        comp = record.get(key, 'N/A')
        print(f"  {key}: {comp[:80]}...")

    ranking_keys = [k for k in record.keys() if 'rank' in k.lower() or 'score' in k.lower()]
    for key in ranking_keys:
        ranking = record.get(key, 'N/A')
        print(f"  {key}: {ranking}")
    print()

# ============================================================================
# 최종 분석 및 권장사항
# ============================================================================

print("="*80)
print("【 최종 분석 및 권장사항 】\n")

print("데이터 품질 요약:")
print(f"  • 총 레코드: {len(rm_data)}건")
print(f"  • Completion 개수: {list(completion_count_patterns.keys())}")
print(f"  • 오류 총합: {sum(errors.values())}건 ({sum(errors.values())/len(rm_data)*100:.1f}%)\n")

print("정제 시 확인사항:")
print("  1. Missing fields 제거 (prompt, completion, ranking 없는 것)")
print("  2. Ranking-completion 개수 불일치 수정")
print("  3. 파싱 오류 (token 문자열) 제거")
print("  4. Ranking 순서 검증 (첫 번째가 최고점인지)")
print("  5. 극단적으로 짧은 completion 검토\n")

print("SFT와의 관계:")
print("  • RM 데이터의 prompt가 SFT 데이터와 겹치는가?")
print("  • RM completion 중 일부가 SFT completion과 같은가?")
print("  → 정제 시 중복 검토 필요")

RM (Reward Model) 데이터 EDA

【 1단계: 데이터 로드 및 기본 통계 】

✓ kochatgpt_1_RM.jsonl 로드 성공
  총 10220건의 레코드

【 2단계: 데이터 구조 분석 】

첫 1개 레코드 구조:

Keys: ['prompt', 'completion_0', 'completion_1', 'completion_2', 'ranking']

  prompt: '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'...
  completion_0: 'Allow me to answer your question. I know that you are curious about me.'...
  completion_1: '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.'...
  completion_2: '라이언에게 말했다.'...
  ranking: list (길이: 3)

【 3단계: Completion 개수 분포 】

  3개 completion 레코드: 10220건 (100.0%)

【 4단계: Ranking/Score 구조 분석 】

Ranking/Score 필드 분포:
  ranking (list): 10220건

【 5단계: Score 분석 】

분석 대상 필드: 'ranking'

Score 범위: 0 ~ 2
평균: 1.0
중앙값: 1.0
표준편차: 0.8

【 Score 분포 분석 】

각 레코드 내 score 차이 분석:
  • 최대 차이 평균: 2.00
  • 최대 차이 범위: 2 ~ 2

  • 차이가 0인 경우 (구분 불가): 0건 (0.0%)
    → 이런 경우는 정제 필요할 수 있음

【 Score 유형 분석 】

  • 정수형: 300건 (100.0%)
  • 실수형: 0건 (0.0%)
  • 범위 유형: 1-5점 척도 (범위: 2)

【 Score 일관성 검사 】

  • 모든 completion의 score가 같은 경우: 0건
    

In [2]:
# ============================================================================
# 오류 유형별 샘플 확인 코드
# ============================================================================

# 1. missing_prompt 샘플 (prompt가 없거나 0자인 경우)
missing_p_samples = [
    (idx, r) for idx, r in enumerate(rm_data)
    if 'prompt' not in r or not r['prompt']
]

print(f"🔍 [1] missing_prompt 샘플 (총 {len(missing_p_samples)}건 중 최대 3개)")
print("=" * 70)
for i, (idx, r) in enumerate(missing_p_samples[:3], 1):
    print(f"{i}. [Index {idx}] Keys: {list(r.keys())}")
    print(f"   Record content: {r}")
    print("-" * 70)

print("\n" + "=" * 70 + "\n")

# 2. empty_completion 샘플 (completion 필드 중 하나라도 비어있는 경우)
empty_c_samples = []
for idx, r in enumerate(rm_data):
    comp_keys = [k for k in r.keys() if 'completion' in k.lower()]
    for k in comp_keys:
        if not r.get(k) or not isinstance(r.get(k), str):
            empty_c_samples.append((idx, k, r))
            break

print(f"🔍 [2] empty_completion 샘플 (총 {len(empty_c_samples)}건 중 최대 3개)")
print("=" * 70)
for i, (idx, empty_key, r) in enumerate(empty_c_samples[:3], 1):
    print(f"{i}. [Index {idx}] 문제 필드: '{empty_key}'")
    print(f"   prompt: {r.get('prompt', 'N/A')[:60]}...")
    print(f"   값: {repr(r.get(empty_key))}")
    print("-" * 70)

print("\n" + "=" * 70 + "\n")

# 3. parsing_error 샘플 ('token' 문자열이 포함된 경우)
parsing_e_samples = []
for idx, r in enumerate(rm_data):
    # string 형태의 value 중 'token'이 들어간 항목 수집
    token_fields = [k for k, v in r.items() if isinstance(v, str) and 'token' in v]
    if token_fields:
        parsing_e_samples.append((idx, token_fields, r))

print(f"🔍 [3] parsing_error 샘플 (총 {len(parsing_e_samples)}건 중 최대 3개)")
print("=" * 70)
for i, (idx, fields, r) in enumerate(parsing_e_samples[:3], 1):
    print(f"{i}. [Index {idx}] 발견 필드: {fields}")
    print(f"   prompt: {r.get('prompt', 'N/A')[:60]}...")
    for f_name in fields:
        # 'token' 부근 앞뒤 100자 잘라서 출력
        val = r[f_name]
        pos = val.find('token')
        start = max(0, pos - 30)
        end = min(len(val), pos + 70)
        print(f"   [{f_name} 발췌]: ...{repr(val[start:end])}...")
    print("-" * 70)

🔍 [1] missing_prompt 샘플 (총 3건 중 최대 3개)
1. [Index 5930] Keys: ['prompt', 'completion_0', 'completion_1', 'completion_2', 'ranking']
   Record content: {'prompt': '', 'completion_0': 'Sorry, as an AI language model, I need more information about what you are asking. Please provide me with more context so that I can assist you better.', 'completion_1': 'The first step in any new project is to choose a goal. There are many different types of goals, but some common goals are to achieve a goal, to work towards a goal, and to learn from a goal.\n\nThere are many different ways to achieve a goal, but some common goals are to achieve a goal, to work towards a goal, and to learn from a goal.', 'completion_2': 'package com.example.demo.controller;\n\nimport com.example.demo.model.User;\nimport com.example.demo.service.UserService;\nimport org.springframework.beans.factory.annotation.Autowired;\nimport org.springframework.web.bind.annotation.*;\n\nimport java.util.List;\n\n@RestController\n@Reques

# 【 RM (Reward Model) 데이터 분석 및 정제 전략 - 최종 】

1. 품질 이슈 현황 및 원인 분석
   - Missing Prompt (3건):
     • Prompt가 비어있는 데이터로, 질의-답변 맥락 학습이 불가능함.
   - Empty Completion (30건):
     • 일부 레코드의 특정 Completion 필드가 빈 문자열('')인 현상.
   - Parsing Error / Dict 파싱 이상 (431건):
     • 파싱 과정 오류로 인해 Completion 문장 끝에 `,'token': 숫자}` 패턴이 노출된 현상.

2. 항목별 정제 규칙 및 처리 전략

   1) Missing Prompt (3건) ➔ 【전량 제거】
        + 질문이 없으면 보상 평가 자체가 불가능하므로 대상 레코드 삭제.  

   2)  Empty Completion (30건) ➔ 【유지 (최하점 Negative Sample 활용)】
       + 레코드 내 다른 Completion들은 정상 텍스트를 포함하고 있음.  
       + 빈 답변('')은 RM 관점에서 가장 완성도가 떨어지는 최악의 응답에 해당하므로,
         <b>최하점(Lowest Rank/Score)을 부여받는 음성 샘플(Negative Sample)</b>로 간주하여 삭제 없이 유지함.  
       + (효과: 모델이 응답을 아예 생성하지 않거나 누락하는 행위에 강력한 감점 보상을 학습하게 됨)  

   3) Parsing Error (431건) ➔ 【정규표현식 전처리 후 원본 복원】
       + SFT 정제 전략과 동일하게 정규식(`re.sub`)을 적용하여 문장 끝의 `,'token':\s*\d+\}?` 패턴만 잘라냄.
       + 답변 본문 자체는 유효하므로 원본 텍스트를 깨끗하게 복원하여 데이터 손실 없이 살림.

3. 종합 평가 및 데이터 손실률
   - 최종 제거 대상: Missing Prompt 3건
   - 최종 수정/유지 대상: Empty Completion 30건(유지), Parsing Error 431건(텍스트 정제)
   - 데이터 손실률: 전체 10,220건 중 약 0.03%(3건)만 제거되어, RM 학습용 비교 데이터의 규모와 다양성을 최대한으로 보존함.